In [1]:
import backtrader as bt
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.tsa.stattools import coint, adfuller
import yfinance as yf
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

class PairsTradingStrategy(bt.Strategy):
    """
    配對交易策略
    基於 Z-score 的均值回歸策略
    """
    params = (
        ('lookback', 20),          # 計算均值和標準差的回望期
        ('entry_threshold', 1.0),   # 進場 Z-score 閾值
        ('exit_threshold', 0.0),    # 出場 Z-score 閾值
        ('stop_loss', 3.0),        # 停損 Z-score 閾值
        ('size', 1000),            # 基礎交易規模
    )
    
    def __init__(self):
        # 獲取兩個資產的數據
        self.data0 = self.datas[0]  # 資產 A
        self.data1 = self.datas[1]  # 資產 B
        
        # 計算對沖比率（使用簡單線性回歸）
        self.hedge_ratio = bt.indicators.OLS_Slope_InterceptN(
            self.data0.close, 
            self.data1.close, 
            period=self.p.lookback
        ).slope
        
        # 計算價差
        self.spread = self.data0.close - self.hedge_ratio * self.data1.close
        
        # 計算 Z-score
        self.spread_mean = bt.indicators.SMA(self.spread, period=self.p.lookback)
        self.spread_std = bt.indicators.StdDev(self.spread, period=self.p.lookback)
        self.zscore = (self.spread - self.spread_mean) / self.spread_std
        
        # 記錄當前持倉狀態
        self.in_position = False
        self.position_type = None  # 'long_spread' 或 'short_spread'
        
    def next(self):
        # 檢查是否有足夠的數據
        if len(self) < self.p.lookback:
            return
            
        # 獲取當前 Z-score
        current_zscore = self.zscore[0]
        
        # 如果沒有持倉
        if not self.in_position:
            # Z-score > +1：做空價差（做空 A，做多 B）
            if current_zscore > self.p.entry_threshold:
                size_a = self.p.size / self.data0.close[0]
                size_b = self.p.size * abs(self.hedge_ratio[0]) / self.data1.close[0]
                
                self.sell(data=self.data0, size=size_a)
                self.buy(data=self.data1, size=size_b)
                
                self.in_position = True
                self.position_type = 'short_spread'
                self.log(f'做空價差: Z-score={current_zscore:.2f}')
                
            # Z-score < -1：做多價差（做多 A，做空 B）
            elif current_zscore < -self.p.entry_threshold:
                size_a = self.p.size / self.data0.close[0]
                size_b = self.p.size * abs(self.hedge_ratio[0]) / self.data1.close[0]
                
                self.buy(data=self.data0, size=size_a)
                self.sell(data=self.data1, size=size_b)
                
                self.in_position = True
                self.position_type = 'long_spread'
                self.log(f'做多價差: Z-score={current_zscore:.2f}')
                
        # 如果有持倉
        else:
            # 檢查停損條件
            if abs(current_zscore) > self.p.stop_loss:
                self.close_all_positions()
                self.log(f'停損平倉: Z-score={current_zscore:.2f}')
                
            # 檢查平倉條件（Z-score 回歸到 0 附近）
            elif abs(current_zscore) <= abs(self.p.exit_threshold):
                self.close_all_positions()
                self.log(f'獲利平倉: Z-score={current_zscore:.2f}')
                
    def close_all_positions(self):
        """平掉所有持倉"""
        self.close(data=self.data0)
        self.close(data=self.data1)
        self.in_position = False
        self.position_type = None
        
    def log(self, txt, dt=None):
        """記錄函數"""
        dt = dt or self.datas[0].datetime.date(0)
        print(f'{dt.isoformat()} {txt}')

class PairsTradingAnalyzer(bt.Analyzer):
    """
    配對交易分析器
    計算策略績效指標
    """
    def __init__(self):
        self.trades = []
        self.pnl = []
        
    def notify_trade(self, trade):
        if trade.isclosed:
            self.trades.append(trade)
            self.pnl.append(trade.pnl)
            
    def get_analysis(self):
        total_trades = len(self.trades)
        if total_trades == 0:
            return {}
            
        winning_trades = sum(1 for trade in self.trades if trade.pnl > 0)
        total_pnl = sum(self.pnl)
        
        return {
            'total_trades': total_trades,
            'winning_trades': winning_trades,
            'losing_trades': total_trades - winning_trades,
            'win_rate': winning_trades / total_trades if total_trades > 0 else 0,
            'total_pnl': total_pnl,
            'avg_pnl': total_pnl / total_trades if total_trades > 0 else 0,
            'max_win': max(self.pnl) if self.pnl else 0,
            'max_loss': min(self.pnl) if self.pnl else 0,
        }

def test_cointegration(data1, data2):
    """
    測試兩個時間序列的協整性
    """
    # 計算相關係數
    correlation = np.corrcoef(data1, data2)[0, 1]
    
    # 進行協整檢驗
    score, pvalue, _ = coint(data1, data2)
    
    # ADF 檢驗（檢查價差的平穩性）
    spread = data1 - (data1.cov(data2) / data2.var()) * data2
    adf_result = adfuller(spread)
    
    return {
        'correlation': correlation,
        'coint_pvalue': pvalue,
        'adf_statistic': adf_result[0],
        'adf_pvalue': adf_result[1],
        'is_cointegrated': pvalue < 0.05 and adf_result[1] < 0.05
    }

def run_pairs_trading_backtest(symbol1, symbol2, start_date, end_date, initial_cash=100000):
    """
    執行配對交易回測
    
    Parameters:
    -----------
    symbol1, symbol2: str, 股票代碼
    start_date, end_date: str, 回測期間
    initial_cash: float, 初始資金
    """
    # 下載數據
    print(f"下載數據: {symbol1} 和 {symbol2}")
    data1 = yf.download(symbol1, start=start_date, end=end_date, progress=False)
    data2 = yf.download(symbol2, start=start_date, end=end_date, progress=False)
    
    # 確保數據對齊
    data1 = data1.loc[data1.index.isin(data2.index)]
    data2 = data2.loc[data2.index.isin(data1.index)]
    
    # 測試協整性
    print("\n測試協整性...")
    coint_results = test_cointegration(
        data1['Close'].values, 
        data2['Close'].values
    )
    
    print(f"相關係數: {coint_results['correlation']:.3f}")
    print(f"協整檢驗 p-value: {coint_results['coint_pvalue']:.3f}")
    print(f"ADF 檢驗 p-value: {coint_results['adf_pvalue']:.3f}")
    print(f"是否協整: {coint_results['is_cointegrated']}")
    
    if not coint_results['is_cointegrated']:
        print("\n警告：這對資產可能不適合配對交易！")
    
    # 創建 Cerebro 引擎
    cerebro = bt.Cerebro()
    
    # 添加數據
    data1_bt = bt.feeds.PandasData(dataname=data1)
    data2_bt = bt.feeds.PandasData(dataname=data2)
    cerebro.adddata(data1_bt, name=symbol1)
    cerebro.adddata(data2_bt, name=symbol2)
    
    # 添加策略
    cerebro.addstrategy(PairsTradingStrategy)
    
    # 添加分析器
    cerebro.addanalyzer(PairsTradingAnalyzer, _name='pairs_analysis')
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    
    # 設置初始資金
    cerebro.broker.setcash(initial_cash)
    
    # 設置手續費
    cerebro.broker.setcommission(commission=0.001)
    
    # 運行回測
    print(f"\n開始回測...")
    print(f"初始資金: ${initial_cash:,.2f}")
    
    results = cerebro.run()
    
    # 獲取結果
    strat = results[0]
    final_value = cerebro.broker.getvalue()
    
    print(f"\n回測結果:")
    print(f"最終資金: ${final_value:,.2f}")
    print(f"總收益: ${final_value - initial_cash:,.2f}")
    print(f"收益率: {((final_value - initial_cash) / initial_cash) * 100:.2f}%")
    
    # 獲取分析結果
    pairs_analysis = strat.analyzers.pairs_analysis.get_analysis()
    sharpe = strat.analyzers.sharpe.get_analysis()
    dd = strat.analyzers.drawdown.get_analysis()
    returns = strat.analyzers.returns.get_analysis()
    
    print(f"\n交易統計:")
    print(f"總交易次數: {pairs_analysis.get('total_trades', 0)}")
    print(f"獲利交易: {pairs_analysis.get('winning_trades', 0)}")
    print(f"虧損交易: {pairs_analysis.get('losing_trades', 0)}")
    print(f"勝率: {pairs_analysis.get('win_rate', 0)*100:.2f}%")
    print(f"平均每筆損益: ${pairs_analysis.get('avg_pnl', 0):.2f}")
    
    print(f"\n風險指標:")
    print(f"夏普比率: {sharpe.get('sharperatio', 'N/A')}")
    print(f"最大回撤: {dd.get('max', {}).get('drawdown', 0):.2f}%")
    print(f"年化收益率: {returns.get('rnorm100', 0):.2f}%")
    
    # 繪製圖表
    cerebro.plot(style='candlestick', barup='green', bardown='red')
    
    return results

In [2]:
results = run_pairs_trading_backtest(
        symbol1 = "KO",      # 可口可樂
        symbol2 = "PEP",     # 百事可樂
        start_date = "2022-01-01",
        end_date = "2023-12-31",
        initial_cash=100000
    )

下載數據: KO 和 PEP

測試協整性...


AttributeError: 'numpy.ndarray' object has no attribute 'cov'

In [27]:
backtester.calculate_performance_metrics()

AttributeError: 'PairsTradingBacktester' object has no attribute 'results'

In [26]:
backtester.test_results()

AttributeError: 'PairsTradingBacktester' object has no attribute 'test_results'

In [7]:
pip install backtrader

Note: you may need to restart the kernel to use updated packages.
